In [ ]:
#Search run summary (best FIDs per run)
import os
import pandas as pd

search_dir = '/path/to/thesis_project/models/cgan/eurosat/seed_42/search/'
SAVE_DIR   = '/path/to/thesis_project/inspection/cgan/eurosat/seed_42/'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"{'Run':<30} {'Epochs':>8} {'Best FID':>10} {'Best Epoch':>12}")
print('-' * 65)

for run in sorted(os.listdir(search_dir)):
    run_path = os.path.join(search_dir, run)
    if not os.path.isdir(run_path):
        continue
    fid_path = os.path.join(run_path, 'fid_scores.csv')
    if not os.path.isfile(fid_path):
        continue
    fid_df = pd.read_csv(fid_path)
    n_epochs = len(pd.read_csv(os.path.join(run_path, 'history.csv')))
    best_idx = fid_df['fid'].idxmin()
    best_fid = fid_df['fid'].iloc[best_idx]
    best_ep  = fid_df['epoch'].iloc[best_idx]
    print(f"{run:<30} {n_epochs:>8} {best_fid:>10.2f} {best_ep:>12}")

In [ ]:
#FID vs Epoch per search run

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

for run in sorted(os.listdir(search_dir)):
    run_path = os.path.join(search_dir, run)
    fid_path = os.path.join(run_path, 'fid_scores.csv')
    if not os.path.isfile(fid_path):
        continue
    fid_df = pd.read_csv(fid_path)
    ax.plot(fid_df['epoch'], fid_df['fid'], marker='o', markersize=3, label=run)

ax.set_title('FID vs Epoch — All Search Runs')
ax.set_xlabel('Epoch')
ax.set_ylabel('FID')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eurosat_search_fid.png'), dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

best_run_name, best_run_df, best_run_fid = None, None, float('inf')

for run in sorted(os.listdir(search_dir)):
    fid_path = os.path.join(search_dir, run, 'fid_scores.csv')
    if not os.path.isfile(fid_path):
        continue
    fid_df = pd.read_csv(fid_path)
    if fid_df['fid'].min() < best_run_fid:
        best_run_fid  = fid_df['fid'].min()
        best_run_df   = fid_df
        best_run_name = run

ax.plot(best_run_df['epoch'], best_run_df['fid'], marker='o', markersize=3, label=best_run_name)

ax.set_title(f'FID vs Epoch — Best Run ({best_run_name})')
ax.set_xlabel('Epoch')
ax.set_ylabel('FID')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eurosat_best_run_fid.png'), dpi=150)
plt.show()

In [ ]:
final_dir = '/path/to/thesis_project/models/cgan/eurosat/seed_42/final/'
history = pd.read_csv(os.path.join(final_dir, 'history.csv'))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Training Curves — EuroSAT cGAN Final', fontsize=13)

axes[0, 0].plot(history['c_loss'], color='steelblue')
axes[0, 0].set_title('Critic Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(history['c_wass_loss'], color='orange')
axes[0, 1].set_title('Critic Wasserstein Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(history['g_loss'], color='green')
axes[1, 0].set_title('Generator Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(history['c_gp'], color='red')
axes[1, 1].set_title('Gradient Penalty')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eurosat_final_loss.png'), dpi=150)
plt.show()

In [ ]:
#Epoch grids one by one + save combined progression

from PIL import Image

checkpoints = [50, 100, 150, 200, 250]

for ep in checkpoints:
    path = f'{final_dir}/epoch_{ep:03d}/generated_epoch_{ep:03d}.png'
    img = Image.open(path)
    w, h = img.size
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.title(f'EuroSAT cGAN — Epoch {ep}  |  Grid size: {w}×{h}px', fontsize=11)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Combined progression figure
fig, axes = plt.subplots(1, len(checkpoints), figsize=(4 * len(checkpoints), 4))
for ax, ep in zip(axes, checkpoints):
    path = f'{final_dir}/epoch_{ep:03d}/generated_epoch_{ep:03d}.png'
    img = Image.open(path)
    ax.imshow(img)
    ax.set_title(f'Epoch {ep}', fontsize=10)
    ax.axis('off')

plt.suptitle('EuroSAT cGAN — Training Progression', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eurosat_training_progression.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data  = np.load('/path/to/thesis_project/data/synthetic/seed_42/cgan/eurosat/eurosat_synthetic.npz')
X_syn = data['X_train']
y_syn = data['y_train']

class_names = ['AnnualCrop', 'Forest', 'HerbaceousVeg', 'Highway', 'Industrial',
               'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']

SAMPLES_PER_CLASS = 5
rng = np.random.default_rng(seed=42)

fig, axes = plt.subplots(10, SAMPLES_PER_CLASS, figsize=(SAMPLES_PER_CLASS * 2, 22))

for cls in range(10):
    idx    = np.where(y_syn == cls)[0]
    chosen = rng.choice(idx, size=SAMPLES_PER_CLASS, replace=False)
    for j, i in enumerate(chosen):
        axes[cls, j].imshow(X_syn[i])
        axes[cls, j].axis('off')
    axes[cls, 0].set_title(class_names[cls], fontsize=8, loc='left')

plt.suptitle('EuroSAT cGAN — Synthetic Samples', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eurosat_synthetic_samples.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to: {SAVE_DIR}')